# 用 Python 分析宁德时代：收益率、波动率与最大回撤

**对象**：希望学习金融数据分析、具备少量 Python 基础的读者。

**标的**：宁德时代（深交所，Yahoo Finance 代码 `300750.SZ`）。

**学习目标**：完成本教程后，你将能够：

- 下载并检查股票历史复权价格；
- 计算日收益率、区间累计收益率和年化复合收益率；
- 计算日波动率与年化波动率；
- 计算回撤序列、最大回撤及其峰值与谷底日期；
- 正确解读收益和风险指标的局限性。

> 默认分析最近 5 年。数据来自 Yahoo Finance，仅供学习；正式投资研究应与交易所公告或专业数据源交叉核验。


## 学习路线

1. 准备环境并设置参数
2. 下载和检查复权收盘价
3. 计算历史收益率
4. 计算年化波动率
5. 计算最大回撤
6. 汇总结果并绘图
7. 完成练习：封装指标函数


## 1. 准备环境

本教程的核心计算只依赖 `pandas` 和 `numpy`；数据下载使用 Python 标准库。绘图为可选功能，如果下一格提示缺少绘图库，可取消注释并运行 `%pip install matplotlib`，然后重启内核。


In [ ]:
# 如需运行后面的可选图表且缺少 matplotlib，取消下一行开头的 # 后执行：
# %pip install -q matplotlib

from __future__ import annotations

import json
import math
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 2. 下载最近 5 年历史价格

`auto_adjust=True` 会使用经过分红、拆股等因素调整后的价格。计算长期收益时，应优先使用复权价格，而不是未经调整的收盘价。


In [ ]:
TICKER = "300750.SZ"
STOCK_NAME = "宁德时代"
RANGE = "5y"  # 可改为 1y、2y、3y、5y、10y 或 max
TRADING_DAYS = 252  # A股一年通常约有 242~252 个交易日，这里采用金融分析常用值 252

params = urllib.parse.urlencode(
    {
        "range": RANGE,
        "interval": "1d",
        "events": "div,splits",
        "includeAdjustedClose": "true",
    }
)
url = f"https://query1.finance.yahoo.com/v8/finance/chart/{TICKER}?{params}"
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

try:
    with urllib.request.urlopen(request, timeout=30) as response:
        payload = json.load(response)
except Exception as exc:
    raise RuntimeError("行情下载失败，请检查网络后重试。") from exc

chart = payload.get("chart", {})
if chart.get("error") or not chart.get("result"):
    raise RuntimeError(f"行情接口返回错误：{chart.get('error')}")

result = chart["result"][0]
timestamps = pd.to_datetime(result["timestamp"], unit="s", utc=True)
dates = timestamps.tz_convert("Asia/Shanghai").tz_localize(None).normalize()

adjusted = result.get("indicators", {}).get("adjclose", [{}])[0].get("adjclose")
if adjusted is None:
    raise RuntimeError("接口没有返回复权收盘价。")

close = pd.Series(adjusted, index=dates, dtype="float64", name="Adjusted Close")
close = close.dropna().groupby(level=0).last().sort_index()

if len(close) < 250:
    raise RuntimeError("有效交易日过少，无法进行可靠的年度化计算。")

print(f"股票：{STOCK_NAME} ({TICKER})")
print(f"实际区间：{close.index[0].date()} 至 {close.index[-1].date()}")
print(f"有效交易日：{len(close):,}")
print("\n开头 3 行：")
print(close.to_frame().head(3).to_string())
print("\n末尾 3 行：")
print(close.to_frame().tail(3).to_string())


## 3. 历史收益率

最常见的简单日收益率为：

$$r_t=\frac{P_t}{P_{t-1}}-1$$

我们同时观察三个指标：

- **日收益率**：每个交易日相对前一交易日的涨跌幅；
- **累计收益率**：整个区间内总共上涨或下跌多少；
- **年化复合收益率（CAGR）**：把区间增长折算成每年的复合增长速度。


In [ ]:
daily_return = close.pct_change().dropna().rename("Daily Return")

total_return = close.iloc[-1] / close.iloc[0] - 1
years = (close.index[-1] - close.index[0]).days / 365.25
cagr = (close.iloc[-1] / close.iloc[0]) ** (1 / years) - 1

print(f"区间累计收益率：{total_return:.2%}")
print(f"年化复合收益率：{cagr:.2%}")
print(daily_return.describe().to_frame().T.to_string())


## 4. 波动率

波动率衡量收益率的离散程度，并不区分上涨和下跌。年化波动率常用以下近似：

$$\sigma_{annual}=\sigma_{daily}\sqrt{252}$$

该公式隐含每日收益近似独立、波动率相对稳定等假设，因此它是历史风险的简化描述，不是未来风险保证。


In [ ]:
daily_volatility = daily_return.std(ddof=1)
annualized_volatility = daily_volatility * math.sqrt(TRADING_DAYS)

print(f"日波动率：{daily_volatility:.2%}")
print(f"年化波动率：{annualized_volatility:.2%}")


## 5. 最大回撤

回撤表示资产价格相对此前历史高点下跌了多少：

$$Drawdown_t=\frac{P_t}{\max(P_0,\ldots,P_t)}-1$$

最大回撤是回撤序列中的最小值。它回答的是：**如果在区间内某个高点买入，之后最糟糕时账面亏损有多大？**


In [ ]:
running_peak = close.cummax()
drawdown = (close / running_peak - 1).rename("Drawdown")
max_drawdown = drawdown.min()

trough_date = drawdown.idxmin()
peak_date = close.loc[:trough_date].idxmax()
peak_price = close.loc[peak_date]

after_trough = close.loc[trough_date:]
recovered = after_trough[after_trough >= peak_price]
recovery_date = recovered.index[0] if not recovered.empty else pd.NaT

print(f"最大回撤：{max_drawdown:.2%}")
print(f"回撤起点：{peak_date.date()}")
print(f"回撤谷底：{trough_date.date()}")
print("恢复日期：", recovery_date.date() if pd.notna(recovery_date) else "截至样本末仍未恢复")


## 6. 汇总关键指标

将收益、波动率和回撤放在一起看。高收益并不必然意味着更好的持有体验；同样的收益可能伴随完全不同的波动和回撤路径。


In [ ]:
metrics = pd.DataFrame(
    {
        "数值": [
            total_return,
            cagr,
            daily_return.mean(),
            daily_volatility,
            annualized_volatility,
            max_drawdown,
        ]
    },
    index=[
        "区间累计收益率",
        "年化复合收益率（CAGR）",
        "平均日收益率",
        "日波动率",
        "年化波动率",
        "最大回撤",
    ],
)

print(metrics.to_string(formatters={"数值": "{:.2%}".format}))


In [ ]:
# 可选图表：未安装 matplotlib 时会跳过，不影响核心指标计算。
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("未安装 matplotlib，已跳过图表。核心指标已经计算完成。")
else:
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), constrained_layout=True)

    (close / close.iloc[0] * 100).plot(ax=axes[0], color="#1565C0", linewidth=1.8)
    axes[0].set_title(f"{STOCK_NAME} normalized adjusted price (start = 100)")
    axes[0].set_ylabel("Index level")

    (drawdown * 100).plot(ax=axes[1], color="#C62828", linewidth=1.4)
    axes[1].fill_between(drawdown.index, drawdown.values * 100, 0, color="#EF5350", alpha=0.25)
    axes[1].set_title("Historical drawdown")
    axes[1].set_ylabel("Drawdown (%)")

    (daily_return * 100).plot.hist(ax=axes[2], bins=50, color="#00897B", alpha=0.8)
    axes[2].axvline(daily_return.mean() * 100, color="black", linestyle="--", linewidth=1)
    axes[2].set_title("Distribution of daily returns")
    axes[2].set_xlabel("Daily return (%)")
    plt.show()


### 如何解读

- **CAGR** 描述起点到终点的复合增长速度，但会隐藏中间路径；
- **年化波动率** 越高，代表日常价格变化通常越剧烈；
- **最大回撤** 越负，代表样本内最痛苦的高点到低点跌幅越深；
- 最大回撤高度依赖所选起止日期，不能单独用来预测未来最坏情形。

### 常见误区

1. 使用未复权价格，导致分红或拆股被误认为真实亏损；
2. 把平均日收益率直接乘以 252 当作长期复合收益率；
3. 把历史波动率当成未来必然发生的风险水平；
4. 只看最大回撤数值，不看回撤持续时间和是否恢复；
5. 比较两只股票时使用不同日期区间。


## 7. 练习

请尝试完成以下任务：

1. 把分析窗口从 5 年改为 3 年，比较结果变化；
2. 编写 `risk_return_metrics()`，输入价格序列，输出 CAGR、年化波动率和最大回撤；
3. 下载沪深 300 指数（Yahoo Finance 代码 `000300.SS`），在相同区间与宁德时代比较。

下面提供第 2 题的答案脚手架。可以先自己填写 `TODO`，再查看注释中的提示。


In [ ]:
def risk_return_metrics(prices: pd.Series, trading_days: int = 252) -> pd.Series:
    """根据复权价格序列计算三个核心风险收益指标。"""
    prices = prices.dropna().sort_index()
    if len(prices) < 2:
        raise ValueError("价格序列至少需要两个有效观测值。")

    returns = prices.pct_change().dropna()
    sample_years = (prices.index[-1] - prices.index[0]).days / 365.25
    if sample_years <= 0:
        raise ValueError("日期索引必须覆盖至少一天。")

    result_cagr = (prices.iloc[-1] / prices.iloc[0]) ** (1 / sample_years) - 1
    result_volatility = returns.std(ddof=1) * np.sqrt(trading_days)
    result_drawdown = (prices / prices.cummax() - 1).min()

    return pd.Series(
        {
            "CAGR": result_cagr,
            "Annualized volatility": result_volatility,
            "Maximum drawdown": result_drawdown,
        }
    )

exercise_result = risk_return_metrics(close).to_frame("Value")
print(exercise_result.to_string(formatters={"Value": "{:.2%}".format}))


## 下一步

掌握这三个指标后，可以继续加入沪深 300 基准，计算超额收益、Beta、夏普比率和滚动波动率。正式决策前还应结合公司基本面、估值、行业周期和数据质量，而不能只依赖历史价格。
